In [1]:
#Import bibliotecas
import os
import sys
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, count, col, current_timestamp, from_json, lit, row_number, sum
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.window import Window

In [2]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [4]:
#Iniciando sessão spark
spark = SparkSession.builder \
    .appName("bronze-layer") \
    .config("spark.jars.packages",
            "org.postgresql:postgresql:42.6.0,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "hadoop") \
    .config("spark.sql.catalog.lakehouse.warehouse", "/tmp/warehouse") \
    .getOrCreate()

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.postgresql#postgresql added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d6478473-d26d-4c70-9bd9-59535b6984dc;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 in central
:: resolution report :: resolve 180ms :: artifacts dl 9ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 from central in [default]
	org.checkerframework#checker-qual;3.31.0 from central in [default]
	org.postgresql#postgresql;42.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number

In [5]:
#Variaveis de conexao ao ambiente postgres
conexao = {
           "url": "jdbc:postgresql://postgres:5432/pipeline_transacoes",
           "user": "postgres",
           "password": "1234",
           "driver": "org.postgresql.Driver"
}

In [6]:
#Dados vindo da postgres
df_raw = spark.read.jdbc(
    url=conexao["url"],
    table="staging.transacoes_raw",  # tabela no PostgreSQL
    properties={
        "user": conexao["user"],
        "password": conexao["password"],
        "driver": conexao["driver"]
    }
)

df_raw.select("payload").show(truncate=True)

+-------+
|payload|
+-------+
+-------+



In [7]:
#Criando tabela bronze se não existir
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.bronze")

#Definir schema do payload
schema_payload = StructType([
    StructField("valor", DoubleType(), True),
    StructField("produto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("quantidade", IntegerType(), True),
    StructField("metodo_pagamento", StringType(), True),
])

# Transformar DF raw
df_bronze = (
    df_raw
    .withColumn("data_ingestao", current_timestamp())
    .withColumn("data_particao", to_date(col("data_ingestao")))
    .withColumn("payload_struct", from_json(col("payload"), schema_payload))
    .select(
        "id",
        "payload_struct.*",
        "data_ingestao",
        "data_particao"
    )
)                           
df_bronze.writeTo("lakehouse.bronze.transacoes") \
         .using("iceberg") \
         .partitionedBy("data_particao") \
         .option("merge-schema", "true") \
         .createOrReplace()
    

In [8]:
df_raw = spark.read \
    .format("jdbc") \
    .options(**conexao) \
    .option("dbtable", "staging.transacoes_raw") \
    .load()

In [9]:
print(" Total de registros na RAW (Postgres):")

df_raw.count()

 Total de registros na RAW (Postgres):


0

In [10]:
spark._sc._jvm.java.sql.DriverManager.getConnection(
    conexao["url"],
    conexao["user"],
    conexao["password"]
).createStatement().execute("TRUNCATE TABLE staging.transacoes_raw")

False

In [11]:
#Select tabela transacoes na bronze
spark.sql("SELECT * FROM lakehouse.bronze.transacoes LIMIT 5").show()

+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+
| id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+



In [12]:
spark.sql("""
SELECT data_particao, COUNT(*) as qtd
FROM lakehouse.bronze.transacoes
GROUP BY data_particao
ORDER BY data_particao
""").show()

+-------------+---+
|data_particao|qtd|
+-------------+---+
+-------------+---+



In [13]:
#Validando estrutura tabela bronze
spark.sql("Describe lakehouse.bronze.transacoes").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|                  id|      int|   NULL|
|               valor|   double|   NULL|
|             produto|   string|   NULL|
|           categoria|   string|   NULL|
|          cliente_id|      int|   NULL|
|          quantidade|      int|   NULL|
|    metodo_pagamento|   string|   NULL|
|       data_ingestao|timestamp|   NULL|
|       data_particao|     date|   NULL|
|# Partition Infor...|         |       |
|          # col_name|data_type|comment|
|       data_particao|     date|   NULL|
+--------------------+---------+-------+



In [14]:
#Validando a existencia de dados nulos na bronze

df_bronze.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_bronze.columns
]).show()

+----+-----+-------+---------+----------+----------+----------------+-------------+-------------+
|  id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|
+----+-----+-------+---------+----------+----------+----------------+-------------+-------------+
|NULL| NULL|   NULL|     NULL|      NULL|      NULL|            NULL|         NULL|         NULL|
+----+-----+-------+---------+----------+----------+----------------+-------------+-------------+



In [15]:
#Verificando dados nulos no campo VALOR
spark.sql("SELECT * FROM lakehouse.bronze.transacoes where valor is null").show()

+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+
| id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+



In [16]:
#Validando se a duplicata para possivel tratamento
df_bronze = spark.read.table("lakehouse.bronze.transacoes")

df_bronze.groupBy("id", "data_ingestao") \
    .agg(count("*").alias("qtd")) \
    .filter(col("qtd") > 1) \
    .show()

+---+-------------+---+
| id|data_ingestao|qtd|
+---+-------------+---+
+---+-------------+---+



In [17]:
#  TRANSFORMAÇÕES (SILVER)
# - tratamento de null
# - deduplicação
window = Window.partitionBy("id").orderBy(col("data_ingestao").desc())

df_curated = (
    df_bronze
    .fillna({"valor": 0})
    .withColumn("row_num", row_number().over(window))
    .filter(col("row_num") == 1)
    .drop("row_num")
    .withColumn("preco_unitario", col("valor") / col("quantidade"))
)

# Criar database
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.silver")

# Salvar
df_curated.writeTo("lakehouse.silver.transacoes") \
    .using("iceberg") \
    .partitionedBy("data_particao") \
    .append()

spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
| id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|preco_unitario|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+



In [18]:
spark.sql("""
SELECT id, COUNT(*) 
FROM lakehouse.silver.transacoes
GROUP BY id
HAVING COUNT(*) > 1
""").show()

+---+--------+
| id|count(1)|
+---+--------+
+---+--------+



In [19]:
df_curated.writeTo("lakehouse.silver.transacoes") \
          .partitionedBy("data_particao") \
          .createOrReplace()

In [20]:
print(" Validando duplicatas após tratamento...")

df_curated.groupBy("id", "data_ingestao") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print(" Dados finais na Silver:")
spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

 Validando duplicatas após tratamento...
+---+-------------+-----+
| id|data_ingestao|count|
+---+-------------+-----+
+---+-------------+-----+

 Dados finais na Silver:
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
| id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|preco_unitario|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+



In [21]:
#Atualização na tabela silver (ADD COLUMNS)
df_add = df_curated.withColumn("preco_unitario", (col("valor") / col("quantidade")))
df_add.writeTo("lakehouse.silver.transacoes") \
          .partitionedBy("data_particao") \
          .createOrReplace()

In [22]:
#Validando após alteração
spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
| id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|preco_unitario|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+



In [23]:
!ls /tmp/warehouse/bronze/transacoes/data/data_particao=2026-04-13

00000-2-1f8cec4a-3bb7-4f22-b2d8-55ef61c8e26f-00001.parquet


In [24]:
print(" ANTES do DELETE:")

df_antes = spark.sql("""
SELECT * FROM lakehouse.silver.transacoes
WHERE id = 1
""")

df_antes.show()

 ANTES do DELETE:
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
| id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|preco_unitario|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+



In [25]:
print("Executando DELETE...")

spark.sql("""
DELETE FROM lakehouse.silver.transacoes
WHERE id = 1
""")

spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

Executando DELETE...
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
| id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|preco_unitario|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+



In [26]:
print(" Snapshots da tabela:")

spark.sql("""
SELECT 
  snapshot_id,
  parent_id,
  committed_at,
  operation
FROM lakehouse.silver.transacoes.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)

 Snapshots da tabela:
+-------------------+-------------------+-----------------------+---------+
|snapshot_id        |parent_id          |committed_at           |operation|
+-------------------+-------------------+-----------------------+---------+
|2883974455146477199|8543515022312238438|2026-04-13 00:18:53.493|delete   |
|8543515022312238438|NULL               |2026-04-13 00:18:53.068|append   |
|6148091971532527351|NULL               |2026-04-13 00:18:52.563|append   |
|3863555896720528809|7923372253755527977|2026-04-13 00:18:52.167|append   |
|7923372253755527977|3476206412951127406|2026-04-13 00:17:32.682|delete   |
|3476206412951127406|NULL               |2026-04-13 00:17:32.224|append   |
|8378399288414862512|NULL               |2026-04-13 00:17:31.32 |append   |
|6034475019797629012|1922523702914660616|2026-04-13 00:17:30.74 |append   |
|1922523702914660616|2856879119974465610|2026-04-13 00:17:08.684|append   |
|2856879119974465610|1736188180257189410|2026-04-13 00:14:44.475|d

In [27]:
print(" Time Travel...")

snapshots = spark.sql("""
SELECT snapshot_id, committed_at
FROM lakehouse.silver.transacoes.snapshots
ORDER BY committed_at DESC
""").collect()

if len(snapshots) > 1:
    snapshot_id = snapshots[1]["snapshot_id"]
    
    print(f" Voltando para snapshot: {snapshot_id}")

    spark.sql(f"""
    SELECT * FROM lakehouse.silver.transacoes
    VERSION AS OF {snapshot_id}
    """).show()
else:
    print(" Não há snapshots suficientes para Time Travel")

 Time Travel...
 Voltando para snapshot: 8543515022312238438
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
| id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|preco_unitario|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+--------------+



In [28]:
df_test = spark.range(0, 1000000)

print(" Comparando performance...")

# PARQUET
start = time.time()
df_test.write.mode("overwrite").parquet("/tmp/parquet_test")
print(f" Escrita Parquet: {time.time() - start:.4f}s")

start = time.time()
spark.read.parquet("/tmp/parquet_test").count()
print(f" Leitura Parquet: {time.time() - start:.4f}s")

# ICEBERG
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.silver.perf_test (
    id BIGINT
)
USING iceberg
""")

start = time.time()
df_test.writeTo("lakehouse.silver.perf_test").overwritePartitions()
print(f" Escrita Iceberg: {time.time() - start:.4f}s")

start = time.time()
spark.read.table("lakehouse.silver.perf_test").count()
print(f" Leitura Iceberg: {time.time() - start:.4f}s")

 Comparando performance...


 Escrita Parquet: 1.2843s
 Leitura Parquet: 0.3294s
 Escrita Iceberg: 0.8759s
 Leitura Iceberg: 0.1195s
